[Reference](https://medium.com/but-it-works-on-my-machine/what-is-agentic-development-exactly-55a76066b957$0)

In [1]:
class Agent:
    def __init__(self, llm_model, tools, memory):
        self.llm = llm_model
        self.tools = tools
        self.memory = memory
        self.current_goal = None

    def set_goal(self, goal_description):
        self.current_goal = goal_description
        self.memory.add_event(f"Goal set: {goal_description}")

    def run(self):
        if not self.current_goal:
            print("No goal set. Please set a goal first.")
            return

        while True:
            # 1. Plan: Ask LLM to determine the next step
            prompt = f"Given the goal: '{self.current_goal}', and past observations: {self.memory.get_recent_history()}, what is the next logical step or action to take? Consider available tools: {self.tools.list_available_tools()}. Respond with 'PLAN: <plan>' or 'ACTION: <tool_name>(<args>)'. If done, respond 'DONE'."

            response = self.llm.generate(prompt)
            self.memory.add_event(f"LLM Response: {response}")

            if response.startswith("DONE"):
                print("Goal achieved!")
                break
            elif response.startswith("PLAN:"):
                plan = response[len("PLAN:"):].strip()
                print(f"Agent Plan: {plan}")
            elif response.startswith("ACTION:"):
                action_str = response[len("ACTION:"):].strip()
                tool_name, args_str = action_str.split('(', 1)
                args = eval(args_str[:-1]) # Dangerous in real code, for example only!

                print(f"Executing tool: {tool_name} with args: {args}")
                observation = self.tools.execute_tool(tool_name, args)
                self.memory.add_observation(observation)
                print(f"Observation: {observation}")
            else:
                print(f"Unexpected response: {response}. Reflecting...")
                # 2. Reflect/Self-Correct: Ask LLM why the response was unexpected
                reflection_prompt = f"I received an unexpected response: '{response}'. My goal is '{self.current_goal}'. What went wrong and how should I adjust my next action or plan?"
                reflection = self.llm.generate(reflection_prompt)
                self.memory.add_event(f"Reflection: {reflection}")
                print(f"Agent Reflection: {reflection}")

# Example Tool (conceptual)
class Tool:
    def list_available_tools(self):
        return ["search_web", "write_file", "read_file"]

    def execute_tool(self, tool_name, args):
        if tool_name == "search_web":
            print(f"Searching web for: {args['query']}")
            return f"Search results for {args['query']}..."
        elif tool_name == "write_file":
            print(f"Writing to {args['path']} content: {args['content']}")
            return f"File {args['path']} written."
        return f"Unknown tool: {tool_name}"

# Example Memory (conceptual)
class Memory:
    def __init__(self):
        self.events = []

    def add_event(self, event):
        self.events.append(event)

    def add_observation(self, observation):
        self.events.append(f"OBSERVATION: {observation}")

    def get_recent_history(self, num_events=5):
        return "\n".join(self.events[-num_events:])